In [1]:
import pandas as pd

Reads price in EUR/MWh and converts them to SEK (1 EUR = 10.77 SEK) and EUR/kWh

In [2]:
import numpy as np

# Read CSV from inputs and keep first + fourth columns
da_price = pd.read_csv("inputs/day_ahead_price.csv").iloc[:, [0, 3]].copy()
# Rename columns
da_price.columns = ["datetime", "da_price"]
# From first column, keep only the initial time part
# (works for values like "2025-01-01 00:00:00 - 2025-01-01 00:15:00")
# On DST change days the start time may end with "(CET)" or "(CEST)".
da_price["datetime"] = (
    da_price["datetime"]
    .astype(str)
    .str.split(" - ")
    .str[0]
    .str.strip()
    .str.replace(r"\s*\((CET|CEST)\)\s*$", "", regex=True)
)
# Parse datetime and numeric price
# Source format is dd/mm/yyyy HH:MM:SS, so parse explicitly to avoid month/day swaps.
da_price["datetime"] = pd.to_datetime(
    da_price["datetime"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)
da_price["da_price"] = pd.to_numeric(da_price["da_price"], errors="coerce")

# Convert price (apply on price column, not datetime)
da_price["da_price"] = da_price["da_price"] * 10.77 * 0.001  # e.g. SEK/kWh
# Cleanup
da_price = da_price.dropna(subset=["datetime", "da_price"]).reset_index(drop=True)

# Add realistic 15-minute variation up to and including 30 September,
# while preserving each hour's exact average price.
cutoff = pd.Timestamp("2025-09-30 23:45:00")
mask_cutoff = da_price["datetime"] <= cutoff
rng = np.random.default_rng(42)
rho = 0.55  # mild correlation across adjacent quarters

# Work on a cutoff slice for speed, then write back once.
work = da_price.loc[mask_cutoff, ["datetime", "da_price"]].copy()
work["hour"] = work["datetime"].dt.floor("h")
work["minute"] = work["datetime"].dt.minute

# Keep only complete quarter-hour blocks (00, 15, 30, 45)
# where the original 4 values are identical within each hour.
size_per_hour = work.groupby("hour")["da_price"].transform("size")
nunique_minute = work.groupby("hour")["minute"].transform("nunique")
price_nunique_per_hour = work.groupby("hour")["da_price"].transform("nunique")
valid = (
    (size_per_hour == 4)
    & (nunique_minute == 4)
    & work["minute"].isin([0, 15, 30, 45])
    & (price_nunique_per_hour == 1)
)
work_valid = work.loc[valid].copy()

prices = work_valid["da_price"].to_numpy(copy=True)
hour_groups = work_valid.groupby("hour", sort=False).indices

for pos in hour_groups.values():
    base = prices[pos].mean()
    sigma = max(abs(base) * 0.02, 0.0005)  # small deviations

    noise = np.zeros(4)
    noise[0] = rng.normal(0, sigma)
    innovation_sigma = sigma * np.sqrt(1 - rho**2)
    for i in range(1, 4):
        noise[i] = rho * noise[i - 1] + rng.normal(0, innovation_sigma)

    # Force zero-mean deviations so quarter-hour average equals original hourly value.
    noise = noise - noise.mean()
    prices[pos] = base + noise

work_valid["da_price"] = prices
da_price.loc[work_valid.index, "da_price"] = work_valid["da_price"].to_numpy()

# Export processed dataset
da_price.to_excel("FerryTerminal_coding/inputs/dayahedprices_15min.xlsx", index=False)
da_price.to_excel("CargoTerminal_coding/inputs/dayahedprices_15min.xlsx", index=False)

da_price.head()

,datetime,da_price
0,2025-01-01 00:00:00,0.034111
1,2025-01-01 00:15:00,0.033426
2,2025-01-01 00:30:00,0.034068
3,2025-01-01 00:45:00,0.034528
4,2025-01-01 01:00:00,0.026369


In [3]:
da_price.tail()

,datetime,da_price
35035,2025-12-31 22:45:00,0.672156
35036,2025-12-31 23:00:00,0.406783
35037,2025-12-31 23:15:00,0.390628
35038,2025-12-31 23:30:00,0.355410
35039,2025-12-31 23:45:00,0.323531
